In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder.appName("CarClassification")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

cols = ["buying", "maintenance", "doors", "persons", "lug_boot", "safety", "car_type"]
df = spark.read.csv("data/car_data.csv", header=True).toDF(*cols)
print("Rows:", df.count())
df.printSchema()
df.show(5)

Rows: 1729
root
 |-- buying: string (nullable = true)
 |-- maintenance: string (nullable = true)
 |-- doors: string (nullable = true)
 |-- persons: string (nullable = true)
 |-- lug_boot: string (nullable = true)
 |-- safety: string (nullable = true)
 |-- car_type: string (nullable = true)

+------+------------+-----+-------+--------+------+--------+
|buying| maintenance|doors|persons|lug_boot|safety|car_type|
+------+------------+-----+-------+--------+------+--------+
|buying|maintainence|doors|persons|lug_boot|safety|car_type|
| vhigh|       vhigh|    2|      2|   small|   low|   unacc|
| vhigh|       vhigh|    2|      2|   small|   med|   unacc|
| vhigh|       vhigh|    2|      2|   small|  high|   unacc|
| vhigh|       vhigh|    2|      2|     med|   low|   unacc|
+------+------------+-----+-------+--------+------+--------+
only showing top 5 rows


# Part 2: Exploration

Explanation: First, check the target distribution. This dataset is very unbalanced, which changes how we judge the models.

In [8]:
total = df.count()
(df.groupBy("car_type").count()
   .withColumn("percent", F.round(F.col("count") * 100 / total, 2))
   .orderBy(F.desc("count"))
   .show())

+--------+-----+-------+
|car_type|count|percent|
+--------+-----+-------+
|   unacc| 1210|  69.98|
|     acc|  384|  22.21|
|    good|   69|   3.99|
|   vgood|   65|   3.76|
|car_type|    1|   0.06|
+--------+-----+-------+



You should see roughly unacc ≈ 70%, acc ≈ 22%, good ≈ 4%, vgood ≈ 4%. A model that always answers “unacc” would already get 70% accuracy, so accuracy alone is not enough.

Explanation: Check for missing values, then see how safety relates to the target.

In [9]:
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()
df.crosstab("safety", "car_type").show()
df.crosstab("persons", "car_type").show()

+------+-----------+-----+-------+--------+------+--------+
|buying|maintenance|doors|persons|lug_boot|safety|car_type|
+------+-----------+-----+-------+--------+------+--------+
|     0|          0|    0|      0|       0|     0|       0|
+------+-----------+-----+-------+--------+------+--------+

+---------------+---+--------+----+-----+-----+
|safety_car_type|acc|car_type|good|unacc|vgood|
+---------------+---+--------+----+-----+-----+
|            low|  0|       0|   0|  576|    0|
|         safety|  0|       1|   0|    0|    0|
|           high|204|       0|  30|  277|   65|
|            med|180|       0|  39|  357|    0|
+---------------+---+--------+----+-----+-----+

+----------------+---+--------+----+-----+-----+
|persons_car_type|acc|car_type|good|unacc|vgood|
+----------------+---+--------+----+-----+-----+
|            more|186|       0|  33|  322|   35|
|         persons|  0|       1|   0|    0|    0|
|               2|  0|       0|   0|  576|    0|
|               4|19

# Part 3: Preparing the features

Explanation: As in the article:

StringIndexer turns each text value into a number.
VectorAssembler combines the 6 columns into one features vector.

We use a Pipeline to chain these steps. handleInvalid="keep" avoids errors if a category appears only in the test set.

In [10]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler

feature_cols = ["buying", "maintenance", "doors", "persons", "lug_boot", "safety"]
idx_cols = [c + "_idx" for c in feature_cols]

label_indexer   = StringIndexer(inputCol="car_type", outputCol="label")   # most frequent class -> 0
feature_indexer = StringIndexer(inputCols=feature_cols, outputCols=idx_cols, handleInvalid="keep")
assembler       = VectorAssembler(inputCols=idx_cols, outputCol="features")

prep = Pipeline(stages=[label_indexer, feature_indexer, assembler]).fit(df)
data = prep.transform(df).select("features", "label", "car_type")
data.show(5, truncate=False)

# Which number stands for which class?
labels = prep.stages[0].labels
print({i: l for i, l in enumerate(labels)})

+-------------------------+-----+--------+
|features                 |label|car_type|
+-------------------------+-----+--------+
|[4.0,4.0,4.0,3.0,3.0,3.0]|4.0  |car_type|
|[3.0,3.0,0.0,0.0,2.0,1.0]|0.0  |unacc   |
|[3.0,3.0,0.0,0.0,2.0,2.0]|0.0  |unacc   |
|[3.0,3.0,0.0,0.0,2.0,0.0]|0.0  |unacc   |
|[3.0,3.0,0.0,0.0,1.0,1.0]|0.0  |unacc   |
+-------------------------+-----+--------+
only showing top 5 rows
{0: 'unacc', 1: 'acc', 2: 'good', 3: 'vgood', 4: 'car_type'}


In [11]:
train_df, test_df = data.randomSplit([0.8, 0.2], seed=42)
train_df.cache(); test_df.cache()
print("Train:", train_df.count(), "| Test:", test_df.count())

Train: 1432 | Test: 297


# Part 4: Evaluation function

Explanation: There are 4 classes, so we use MulticlassClassificationEvaluator. AUC only works for 2 classes, which is why the article’s “Area Under ROC” label is wrong. We report:

1. Accuracy: % of predictions that are correct.
2. F1 (weighted): balances precision and recall, weighted by class size.
3. Recall per class: shows whether the rare classes (good, vgood) are actually detected.
4. Confusion matrix

In [12]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
ev = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

def evaluate(pred, name):
    acc = ev.evaluate(pred, {ev.metricName: "accuracy"})
    f1  = ev.evaluate(pred, {ev.metricName: "f1"})
    print(f"=== {name} ===\naccuracy: {acc:.4f} | weighted F1: {f1:.4f}")
    for i, l in enumerate(labels):
        r = ev.evaluate(pred, {ev.metricName: "recallByLabel", ev.metricLabel: float(i)})
        print(f"  recall {l:>6}: {r:.4f}")
    print("Confusion matrix (rows = true, columns = predicted):")
    pred.groupBy("label").pivot("prediction", [float(i) for i in range(len(labels))]) \
        .count().fillna(0).orderBy("label").show()
    return {"model": name, "accuracy": acc, "f1": f1}

# Part 5: The 3 models from the article
5.1 Logistic Regression

Explanation: This is a linear model. Here the categories are just numbers (0, 1, 2, 3), so the model treats “low < med < high” as a straight line. That’s a poor fit for this problem, so expect weak results. It’s what the article observes too.

In [14]:
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier

lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)
lr_pred = lr.fit(train_df).transform(test_df)
lr_m = evaluate(lr_pred, "Logistic Regression")

=== Logistic Regression ===
accuracy: 0.6599 | weighted F1: 0.6267
  recall  unacc: 0.8592
  recall    acc: 0.2581
  recall   good: 0.0000
  recall  vgood: 0.2727


Py4JJavaError: An error occurred while calling o1321.evaluate.
: java.util.NoSuchElementException: key not found: 4.0
	at scala.collection.immutable.Map$Map4.apply(Map.scala:535)
	at org.apache.spark.mllib.evaluation.MulticlassMetrics.recall(MulticlassMetrics.scala:160)
	at org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator.evaluate(MulticlassClassificationEvaluator.scala:162)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:77)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.base/java.lang.reflect.Method.invoke(Method.java:569)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244)
	at py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
	at py4j.Gateway.invoke(Gateway.java:282)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)